In [ ]:
PROJECT_REF = "main"


def _install_project(project_ref: str):
    import urllib.parse
    import urllib.request

    from google.colab import userdata

    github_token = userdata.get("GITHUB_TOKEN_JLENS_REAS")
    if not github_token:
        raise RuntimeError(
            "Required Colab secret GITHUB_TOKEN_JLENS_REAS is unavailable"
        )

    query = urllib.parse.urlencode({"ref": project_ref})
    bootstrap_url = (
        "https://api.github.com/repos/noamdwc/jlens-reasoning/"
        "contents/scripts/colab_bootstrap.py?" + query
    )
    request = urllib.request.Request(
        bootstrap_url,
        headers={
            "Authorization": f"Bearer {github_token}",
            "Accept": "application/vnd.github.raw+json",
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )
    try:
        with urllib.request.urlopen(request) as response:
            bootstrap_source = response.read().decode("utf-8")
    except Exception:
        raise RuntimeError("Unable to load the Colab bootstrap") from None

    namespace = {}
    exec(
        compile(
            bootstrap_source,
            "scripts/colab_bootstrap.py",
            "exec",
        ),
        namespace,
    )
    return namespace["bootstrap"](
        project_ref=project_ref,
        github_token=github_token,
    )


PROJECT_DIR = _install_project(PROJECT_REF)
del _install_project

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=True)
context

In [ ]:
from experiments.jlens_readout_sanity.experiment import (
    Case,
    ExperimentRuntime,
    InterventionSpec,
    ReadoutSpec,
)

CASES = (
    Case(
        key="spider",
        prompt="The number of legs on the animal that spins webs is",
        expected_answers=("8", "eight"),
        readout=ReadoutSpec(
            concepts=("spider",),
            require_capability_gate=True,
        ),
        intervention=InterventionSpec(
            source_surface=" spider",
            target_surface=" ant",
            target_answers=("6", "six"),
        ),
    ),
    Case(
        key="france_capital",
        prompt="The capital of France is the city of",
        expected_answers=("Paris",),
        readout=ReadoutSpec(("France",), literal_argument="France"),
        intervention=InterventionSpec(" France", " China", ("Beijing",)),
    ),
    Case(
        key="france_language",
        prompt="Most people in France speak",
        expected_answers=("French",),
        readout=ReadoutSpec(("France",), literal_argument="France"),
        intervention=InterventionSpec(" France", " China", ("Chinese",)),
    ),
    Case(
        key="france_continent",
        prompt="France is a country on the continent of",
        expected_answers=("Europe",),
        readout=ReadoutSpec(("France",), literal_argument="France"),
        intervention=InterventionSpec(" France", " China", ("Asia",)),
    ),
    Case(
        key="france_currency",
        prompt="The single-word name for the currency now used in France is the",
        expected_answers=("Euro",),
        readout=ReadoutSpec(("France",), literal_argument="France"),
        intervention=InterventionSpec(" France", " China", ("Yuan",)),
    ),
)

In [ ]:
import importlib.metadata
import subprocess

import jlens
import torch
import transformers

from experiments.jlens_readout_sanity.constants import (
    LENS_FILE,
    LENS_REPO,
    LENS_REVISION,
    MODEL_NAME,
)
from experiments.jlens_readout_sanity.utils import (
    render_sanity_report,
    run_experiment,
    validate_model_lens,
    write_results,
)
from jlens_reasoning.evaluation import GenerationStatus, ModelOutput

causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
).to(context.device)
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(causal_lm, tokenizer)
lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO,
    filename=LENS_FILE,
    revision=LENS_REVISION,
)
validate_model_lens(model, lens)
model, lens

In [ ]:
@torch.inference_mode()
def forward_next_token(input_ids):
    return causal_lm(input_ids=input_ids, use_cache=False).logits[0, -1]


@torch.inference_mode()
def generate_output(prompt: str) -> ModelOutput:
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(context.device)
    generated = causal_lm.generate(
        input_ids=input_ids,
        do_sample=False,
        max_new_tokens=64,
    )
    generated_ids = generated[0, input_ids.shape[1] :].tolist()
    eos_ids = causal_lm.generation_config.eos_token_id
    eos_token_ids = {eos_ids} if isinstance(eos_ids, int) else set(eos_ids or ())
    generation_status = (
        GenerationStatus.COMPLETE
        if generated_ids and generated_ids[-1] in eos_token_ids
        else GenerationStatus.TRUNCATED
    )
    text_ids = (
        generated_ids[:-1]
        if generation_status is GenerationStatus.COMPLETE
        else generated_ids
    )
    return ModelOutput(
        text=tokenizer.decode(text_ids, skip_special_tokens=True),
        token_ids=tuple(generated_ids),
        token_pieces=tuple(
            tokenizer.decode([token_id], clean_up_tokenization_spaces=False)
            for token_id in generated_ids
        ),
        generation_status=generation_status,
        finish_reason=(
            "eos" if generation_status is GenerationStatus.COMPLETE else "length"
        ),
    )


runtime = ExperimentRuntime(
    model=model,
    lens=lens,
    tokenizer=tokenizer,
    unembedding_weight=causal_lm.get_output_embeddings().weight,
    forward_next_token=forward_next_token,
    generate_output=generate_output,
)
result = run_experiment(cases=CASES, runtime=runtime)

In [ ]:
result.provenance = {
    "project_commit": subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "jlens": importlib.metadata.version("jlens"),
}

run_dir = context.runs_dir / "jlens-readout-sanity"
result_path = run_dir / "result.json"
write_results(result_path, result)
print(f"Saved: {result_path}")

In [ ]:
print(render_sanity_report(result))

In [ ]:
if not result.passed:
    raise RuntimeError(
        "Read-and-change sanity checks failed: " + "; ".join(result.failures)
    )

print("All J-Lens read-and-change sanity checks passed.")